# Maile dla analityka: kolejka po przebiegu, podsumowanie kampanii, wynik rundy

Jednorazowa konfiguracja trzech reguł alertu w Cloud Monitoring. Wszystkie działają tak samo: komponent wypisuje do logów jedną linię ze znacznikiem, reguła łapie ją, wycina pola i wysyła maila.

- **„FSL - kolejka dla analityka”**: po pojedynczym przebiegu z `enable_al=True`. Linia `FSL_ANALYST_DIGEST` z komponentu active learning: kolejka przypadków do oznaczenia i link do raportów.
- **„FSL - kampania (zbiorczy)”**: po kampanii foldów (`komorka_kampania.py`). Linia `FSL_CAMPAIGN_DIGEST` z podsumowania kampanii: werdykty bramki, lift z przedziałem ufności, active learning i kolejki ze wszystkich foldów, w jednym mailu.
- **„FSL - runda active learning”**: po rundzie z `al_runda.ipynb`. Linia `FSL_ROUND_DIGEST`: trafność przed i po odpowiedziach analityka, werdykt bramki, następna partia do oznaczenia.

Bez kluczy i bez nowych usług. Adresy biorą się z `notifications.analyst_emails` w `configs/pipeline_config_samld.json`. Każda reguła wysyła najwyżej jeden mail na 5 minut, co przy tych przebiegach niczego nie gubi.

Kolejność: komórka 1 (instalacja), restart kernela, potem komórki 2–4. Ponowne uruchomienie komórek 2–4 aktualizuje istniejące reguły i dodaje brakujące. Komórka 5 to test: wysyła wszystkie trzy maile od razu, bez pipeline'u.

In [ ]:
# KOMORKA 1 — biblioteki (jednorazowo). Po PIERWSZEJ instalacji: Kernel -> Restart, potem od komorki 2.
%pip install -q google-cloud-monitoring google-cloud-logging

In [ ]:
# KOMORKA 2 — config: projekt i adresy analitykow
import json
from pathlib import Path

REPO = Path.home() / "mlops-meta-learning/fsl-projekt"
CFG = json.loads((REPO / "configs/pipeline_config_samld.json").read_text())
PROJECT = CFG["project"]
ANALYST_EMAILS = CFG.get("notifications", {}).get("analyst_emails", [])
assert ANALYST_EMAILS, "brak notifications.analyst_emails w configu"
# konto Google z dostepem do projektu: linki w mailu otwieraja sie jako to konto (?authuser=...)
CONSOLE_ACCOUNT = CFG.get("notifications", {}).get("console_account", ANALYST_EMAILS[0])
print("projekt:", PROJECT, "| mail dla analityka do:", ANALYST_EMAILS, "| linki otwiera konto:", CONSOLE_ACCOUNT)

In [ ]:
# KOMORKA 3 — kanaly powiadomien: jeden kanal e-mail na adres (istniejace sa uzywane ponownie)
from google.cloud import monitoring_v3

PARENT = f"projects/{PROJECT}"
ch_client = monitoring_v3.NotificationChannelServiceClient()
have = {c.labels.get("email_address"): c.name
        for c in ch_client.list_notification_channels(name=PARENT) if c.type_ == "email"}
CHANNELS = []
for addr in ANALYST_EMAILS:
    if addr not in have:
        ch = ch_client.create_notification_channel(
            name=PARENT,
            notification_channel=monitoring_v3.NotificationChannel(
                type_="email", display_name=f"FSL analityk {addr}", labels={"email_address": addr}))
        have[addr] = ch.name
        print("utworzony kanal:", addr)
    CHANNELS.append(have[addr])
print("kanaly:", CHANNELS)

In [ ]:
# KOMORKA 4 — reguly alertu: linia ze znacznikiem w logach pipeline'u -> mail
# Uruchomiona drugi raz aktualizuje istniejace reguly (po nazwie), nie tworzy duplikatow.
from google.protobuf import duration_pb2

AP = monitoring_v3.AlertPolicy

RUN_DOC = r'''**Nowa kolejka przypadków do oznaczenia**

**Otwórz raport active learning:** ${log.extracted_label.report}?authuser={AUTH}

Fold ${log.extracted_label.fold}, model: ${log.extracted_label.model}.
„not-registered” oznacza, że bramka odrzuciła model. Kolejka jest liczona na surowych cechach przypadków, więc obowiązuje niezależnie od werdyktu.

**Do oznaczenia** (model waha się między dwiema typologiami):
${log.extracted_label.queue}

**Typologie w tej rundzie:** ${log.extracted_label.typologies}

**Active learning:** ${log.extracted_label.saving}

**Pliki do oznaczenia** (kolejka.csv z pustą kolumną typologia_analityka, transakcje.csv, README.txt):
${log.extracted_label.files}&authuser={AUTH}
Wypełniony kolejka.csv odłóż do analyst/oznaczone/ pod tą samą ścieżką.

**Raporty** (bramka, explain, active learning z wyjaśnieniem każdego przypadku):
${log.extracted_label.run}&authuser={AUTH}
'''

CAMPAIGN_DOC = r'''**Kampania foldów zakończona** (${log.extracted_label.folds} foldów)

**Bramka:** przeszło ${log.extracted_label.passed}.

**Ile dodaje uczenie metryki** (lift względem losowego, nietrenowanego enkodera; średnia z 95% CI po foldach):
${log.extracted_label.lift}

**Per fold:** ${log.extracted_label.per_fold}

**Active learning** (etykiety potrzebne, żeby dojść do trafności losowego wyboru przy 24): ${log.extracted_label.al_labels}

**Do oznaczenia** (po 2 przypadki z każdego foldu):
${log.extracted_label.queue}

**Pliki do oznaczenia** (folder na każdy fold: kolejka.csv, transakcje.csv, README.txt):
${log.extracted_label.files}&authuser={AUTH}

**Raport zbiorczy i raporty foldów:** ${log.extracted_label.run}&authuser={AUTH}
'''

ROUND_DOC = r'''**Runda ${log.extracted_label.round} active learning zakończona** (fold ${log.extracted_label.fold})

**Bramka:** ${log.extracted_label.verdict}

**Trafność wzorców na zbiorze kontrolnym:** ${log.extracted_label.accuracy}

**Odpowiedzi analityka:** ${log.extracted_label.labels}; ${log.extracted_label.agreement}.

**Otwórz raport rundy:** ${log.extracted_label.report}?authuser={AUTH}

**Następna partia do oznaczenia:**
${log.extracted_label.queue}

**Pliki następnej partii:** ${log.extracted_label.files}&authuser={AUTH}
Wypełniony kolejka.csv odłóż do analyst/oznaczone/ pod tą samą ścieżką.

**Przebieg:** ${log.extracted_label.run}&authuser={AUTH}
'''

RUN_DOC = RUN_DOC.replace("{AUTH}", CONSOLE_ACCOUNT)
CAMPAIGN_DOC = CAMPAIGN_DOC.replace("{AUTH}", CONSOLE_ACCOUNT)
ROUND_DOC = ROUND_DOC.replace("{AUTH}", CONSOLE_ACCOUNT)

SPECS = [
    {"name": "FSL - kolejka dla analityka", "marker": "FSL_ANALYST_DIGEST",
     "keys": ("fold", "model", "typologies", "saving", "queue", "report", "files", "run"),
     "subject": "FSL: kolejka dla analityka, fold ${log.extracted_label.fold}", "doc": RUN_DOC},
    {"name": "FSL - kampania (zbiorczy)", "marker": "FSL_CAMPAIGN_DIGEST",
     "keys": ("folds", "passed", "lift", "per_fold", "al_labels", "queue", "files", "run"),
     "subject": "FSL: kampania zakonczona, bramka ${log.extracted_label.passed}", "doc": CAMPAIGN_DOC},
    {"name": "FSL - runda active learning", "marker": "FSL_ROUND_DIGEST",
     "keys": ("fold", "round", "verdict", "accuracy", "labels", "agreement", "queue", "report", "files", "run"),
     "subject": "FSL: runda ${log.extracted_label.round} active learning, fold ${log.extracted_label.fold}",
     "doc": ROUND_DOC},
]

pol_client = monitoring_v3.AlertPolicyServiceClient()
existing = {p.display_name: p.name for p in pol_client.list_alert_policies(name=PARENT)}
for spec in SPECS:
    policy = AP(
        display_name=spec["name"],
        combiner=AP.ConditionCombinerType.OR,
        conditions=[AP.Condition(
            display_name=f"{spec['marker']} w logach pipeline'u",
            condition_matched_log=AP.Condition.LogMatch(
                # tylko linie ZACZYNAJACE sie od znacznika: Vertex loguje tez wejscia krokow (np. podsumowania
                # kampanii), w ktorych ta sama linia siedzi w srodku JSON-a - tych kopii nie chcemy lapac
                filter=f'resource.type="ml_job"\njsonPayload.message=~"^{spec["marker"]}"',
                label_extractors={k: f'REGEXP_EXTRACT(jsonPayload.message, "{k}=([^|]+)")' for k in spec["keys"]},
            ),
        )],
        alert_strategy=AP.AlertStrategy(
            notification_rate_limit=AP.AlertStrategy.NotificationRateLimit(period=duration_pb2.Duration(seconds=300)),
            auto_close=duration_pb2.Duration(seconds=1800),
        ),
        documentation=AP.Documentation(content=spec["doc"], mime_type="text/markdown", subject=spec["subject"]),
        notification_channels=CHANNELS,
        enabled=True,
    )
    if spec["name"] in existing:
        policy.name = existing[spec["name"]]
        print("zaktualizowana regula:", pol_client.update_alert_policy(alert_policy=policy).name)
    else:
        print("utworzona regula:", pol_client.create_alert_policy(name=PARENT, alert_policy=policy).name)

In [ ]:
# KOMORKA 5 — TEST bez pipeline'u: trzy wpisy do logow wygladajace jak linie z komponentow.
# Trzy maile (kolejka, kampania, runda) powinny przyjsc w ciagu kilku minut (sprawdz tez spam).
# Swiezo utworzona regula zaczyna dzialac po kilku minutach: jesli nic nie przyjdzie, uruchom te komorke ponownie za 10 min.
# Jesli nie przyjda w ~10 minut, wklej mi output tej komorki i status regul z Monitoring -> Alerting.
from google.cloud import logging as gcl
from google.cloud.logging_v2.resource import Resource

TEST_LINES = [
    ("FSL_ANALYST_DIGEST | fold=3 | model=TEST | typologies=Behavioural_Change_2, Gather-Scatter, "
     "Layered_Fan_In, Smurfing | saving=10 labels instead of 24 (TEST) | "
     "queue=#1156 (Gather-Scatter vs Layered_Fan_In); #285 (Gather-Scatter vs Layered_Fan_In) | report=- | files=- | run=-"),
    ("FSL_CAMPAIGN_DIGEST | folds=5 | passed=0/5 (TEST) | lift=-0.012, 95% CI [-0.100, +0.076] | "
     "per_fold=f0 +0.001 FAIL; f1 +0.053 FAIL; f2 -0.056 FAIL; f3 +0.052 FAIL; f4 -0.110 FAIL | "
     "al_labels=f3 10/24 (TEST) | queue=f3: #1156 (Gather-Scatter vs Layered_Fan_In) | files=- | run=-"),
    ("FSL_ROUND_DIGEST | fold=3 | round=1 | verdict=przyjeta, nowe wzorce w uzyciu (TEST) | "
     "accuracy=74,5% -> 81,3% (+6,8 pp) | labels=8 nowych, razem 16 | agreement=analityk poprawil model w 2 z 8 | "
     "queue=#1156 (Gather-Scatter vs Layered_Fan_In) | report=- | files=- | run=-"),
]
logger = gcl.Client(project=PROJECT).logger("fsl-analyst-digest-test")
res = Resource(type="ml_job", labels={"project_id": PROJECT, "job_id": "alert-test", "task_name": "workerpool0-0"})
for line in TEST_LINES:
    logger.log_struct({"message": line}, severity="INFO", resource=res)
    print("wpis testowy zapisany:", line[:70], "...")

Wyłączenie: Monitoring → Alerting → reguła („FSL - kolejka dla analityka”, „FSL - kampania (zbiorczy)” albo „FSL - runda active learning”) → Disable. Kanały e-mail można usunąć w Monitoring → Alerting → Edit notification channels.